In [1]:
R.version

               _                                
platform       x86_64-w64-mingw32               
arch           x86_64                           
os             mingw32                          
crt            ucrt                             
system         x86_64, mingw32                  
status                                          
major          4                                
minor          5.2                              
year           2025                             
month          10                               
day            31                               
svn rev        88974                            
language       R                                
version.string R version 4.5.2 (2025-10-31 ucrt)
nickname       [Not] Part in a Rumble           

In [19]:
library("ape")
library("phytools")
library("nlme")
library("corHMM")
library("geiger")
library("mkcor")
library("OUwie")

Loading required package: GenSA



In [20]:
packageVersion("OUwie")

[1] '2.16'

In [15]:
# example model fitting

data(tworegime)
dat <- data.frame(sp = tree$tip.label, X = sample(c(0, 1), length(tree$tip.label),
    replace = TRUE), Y = sample(c(0, 1), length(tree$tip.label), replace = TRUE),
    FS = rnorm(length(tree$tip.label), 10, 3))
print(head(dat))

  sp X Y        FS
1 t1 1 1  9.123968
2 t2 0 0 13.561094
3 t3 0 1  7.272848
4 t4 1 0 14.297987
5 t5 1 0  4.973740
6 t6 1 0  4.502301


In [25]:
p <- c(0.01670113, 0.39489947, 0.18619839, 1.67259459, 0.16817414)  # my fixed set of parameters
pp_oum <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25, p = p)  # you likely won't use this p argument

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Calculating likelihood from a set of fixed parameters.
[1]  0.01670113  0.39489947  0.18619839 51.67259459 50.16817414


In [17]:
pp_oum


Fit
    lnLTot   lnLDisc   lnLCont     AIC     AICc      BIC nTaxa nPars
 -25.15545 -5.260483 -18.62633 60.3109 61.34539 71.10532    64     5

Legend
  1   2 
"1" "2" 

Regime Rate matrix
           (1)        (2)
(1)         NA 0.01670113
(2) 0.01670113         NA

OU Estimates
             (1)       (2)
alpha  0.3948995 0.3948995
sigma2 0.1861984 0.1861984
theta  1.6725946 0.1681741


Half-life (another way of reporting alpha)
    (1)     (2) 
1.75525 1.75525 

In [18]:
# fitting without p
model <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) 

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


In [35]:
# fit the model to my data
fred_tree <- ape::multi2di(ape::read.tree("../data/chapter2/uphylomaker/fredv3subset_collab_trait_n_states.tre"))
data <- read.csv("../data/chapter2/FREDv3subset/FRED_subset_collab_states_n_species_avg_traits.csv", row.names = "binominal")

In [36]:
data <- data.frame(binominal = as.factor(gsub(rownames(data), pattern = ' ', replacement = '_')), rd = data$F00679, srl = data$F00727, myco = as.factor(data$F00645))
row_indices <- match(fred_tree$tip.label, data$binominal)
all(data$binominal[row_indices] == fred_tree$tip.label)
data <- data[row_indices, ]
all(data$binominal == fred_tree$tip.label) # cool

[1] TRUE

[1] TRUE

In [38]:
head(data)

,binominal,rd,srl,myco
,<fct>,<dbl>,<dbl>,<fct>
39,Anaphalis_aureopunctata,0.136950,479.1550,AM + NM
40,Anaphalis_hancockii,0.178700,319.8579,AM
311,Solidago_decurrens,0.248225,202.8837,AM
129,Doellingeria_scabra,0.211300,219.2008,AM
54,Aster_tataricus,0.197000,297.1100,AM
49,Artemisia_igniaria,0.167200,204.1529,AM


In [39]:
d <- data[, c("binominal", "myco", "srl")]
head(d)

,binominal,myco,srl
,<fct>,<fct>,<dbl>
39,Anaphalis_aureopunctata,AM + NM,479.1550
40,Anaphalis_hancockii,AM,319.8579
311,Solidago_decurrens,AM,202.8837
129,Doellingeria_scabra,AM,219.2008
54,Aster_tataricus,AM,297.1100
49,Artemisia_igniaria,AM,204.1529


In [40]:
model <- OUwie::hOUwie(phy = tree, data = d, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25)

Your phylogeny had node labels, these have been removed.


ERROR: Error in 1:nObs: NA/NaN argument
